# 03 — Paradigma 2: detección de anomalías en RAW-FABRID (bloque B)

**Pregunta:** si sólo tenemos ejemplos de **tela sin defectos**, ¿qué tan bien se pueden detectar y localizar agujeros, manchas y otros defectos? ¿Qué aporta usar características de una CNN preentrenada frente a un método clásico, y qué decisiones de PatchCore importan?

En el bloque A (CISUTAC) el modelo aprende de cajas etiquetadas. Acá ningún método ve un defecto durante el entrenamiento: aprenden cómo es la tela normal y marcan lo que se aparta de eso.

| ID | Método | Idea | Por qué compararlo |
|---|---|---|---|
| **B0** | Estadística local (clásico) | Media y desvío de brillo en una ventana (filtros de caja), comparados con los de la tela normal | Piso sin aprendizaje profundo: conecta con los filtros vistos en la materia |
| **B1** | PaDiM | Una gaussiana por **posición** del mapa de características de una ResNet; anomalía = distancia de Mahalanobis | Supone imágenes alineadas (misma posición = misma parte). Una textura no cumple eso |
| **B2** | PatchCore | Banco de memoria de características **locales** normales; anomalía = distancia al vecino más cercano | No depende de la posición: la hipótesis es que se adapta mejor a una tela |
| **B2-abl** | Ablaciones de PatchCore | backbone, tamaño del coreset, cantidad de normales, resolución | Qué decisiones importan y cuánto cuestan |

**Protocolo** (EDA RAW-6, `reports/raw_fabrid/eda_raw_fabrid.md`):

- Entrenamiento **sólo con `train/good`** (patches de 507 imágenes madre normales).
- Las 202 madres de test se parten **por madre y estratificado por rollo** en *valid* (30 %: elegir variantes y umbral) y *test* (70 %: reporte final, una sola vez). El manifiesto queda en `reports/raw_fabrid/rawB_split_valid_test.csv`.
- Métricas: **AUROC por patch**, **AUROC por píxel** y **AUPRO@0.3**, con intervalo por **bootstrap agrupado por madre** y AUROC por **rollo**.
- Las 2 máscaras huérfanas del EDA son de imágenes high-res, así que no afectan a los patches.

> **AUROC:** probabilidad de que un patch defectuoso reciba más puntaje que uno normal (0,5 = azar, 1 = perfecto). **AUPRO:** cuánto de cada región defectuosa queda marcada mientras se marca poco tejido sano; cada defecto pesa igual, sea grande o chico.

> **Cómo correrlo.** Funciona en CPU. Requiere RAW-FABRID en `data/raw_fabrid/extracted/RAW_FABRID/` (ver `docs/DATASETS.md`) y `pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu` (además de scikit-learn, scipy y opencv). Primero correrlo con `MODO_PRUEBA = True` (minutos). Cada corrida se guarda en `results/raw_fabrid/bloque_b/` y se reutiliza al re-ejecutar (salvo `RECALCULAR = True`).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import anomaly_baseline as ab

pd.set_option("display.max_columns", 30)
print("RAW-FABRID:", ab.RAW)
print("Resultados:", ab.RESULTS)

## 0. Parámetros

PatchCore no se entrena con *backpropagation*: pasa las imágenes normales una vez por la red y guarda características. El costo en CPU depende sobre todo de **cuántas normales** se usan y del **tamaño del banco de memoria**, por eso la cantidad de normales es un parámetro explícito (y una ablación).

In [ ]:
MODO_PRUEBA = True       # True: pocas imágenes, para verificar el pipeline en minutos
RECALCULAR = False       # True: ignora corridas guardadas
EVALUAR_TEST = False     # True sólo al final, una única vez, con la variante ya elegida en valid

SEMILLA = 0
N_NORMALES = 500         # patches train/good usados para ajustar todos los métodos (mismos para todos)
BACKBONE = "resnet18"    # liviano para CPU; WideResNet50 (el del paper) va como ablación
CORESET = 0.01           # fracción del banco de memoria que conserva PatchCore
FRACCION_VALID = 0.3     # madres de test reservadas para elegir variantes

if MODO_PRUEBA:
    N_NORMALES = 40
SUFIJO = "_prueba" if MODO_PRUEBA else ""

def nombre(variante):
    return f"{variante}{SUFIJO}"

In [ ]:
import importlib.metadata as md
for paquete in ("torch", "torchvision", "scikit-learn", "scipy", "opencv-python"):
    try:
        print(f"{paquete:14s} {md.version(paquete)}")
    except md.PackageNotFoundError:
        print(f"{paquete:14s} FALTA")

## 1. Datos y particiones

Se lee el CSV oficial de patches. Cada patch conserva su imagen madre (`parent_raw`) y su rollo para agrupar la incertidumbre.

In [ ]:
patches = ab.load_patches()
splits = ab.split_test_parents(patches, valid_fraction=FRACCION_VALID)

resumen = pd.DataFrame({split: {"patches": len(items), "normales": sum(i["label"] == 0 for i in items),
                                "defectuosos": sum(i["label"] == 1 for i in items),
                                "madres": len({i["parent"] for i in items})}
                        for split, items in splits.items()}).T
resumen

Verificación de independencia: ninguna madre puede estar en dos particiones. También se muestra cómo quedan repartidos los rollos entre valid y test.

In [ ]:
madres = {split: {i["parent"] for i in items} for split, items in splits.items()}
assert not (madres["train"] & madres["valid"]) and not (madres["train"] & madres["test"]) \
       and not (madres["valid"] & madres["test"]), "hay madres compartidas entre particiones"
print("Sin madres compartidas entre train, valid y test.")

pd.crosstab([i["roll"] for s in ("valid", "test") for i in splits[s]],
            [s for s in ("valid", "test") for i in splits[s]], margins=True)

In [ ]:
ab.plot_samples(splits["valid"], n=5);

Conjuntos que efectivamente se usan. Todos los métodos se ajustan con **las mismas** `N_NORMALES` imágenes normales, elegidas con semilla fija.

In [ ]:
train = ab.sample_items(splits["train"], N_NORMALES, seed=SEMILLA)
if MODO_PRUEBA:
    valid = ab.stratified_subset(splits["valid"], 60, 20, seed=SEMILLA)
    test = ab.stratified_subset(splits["test"], 60, 20, seed=SEMILLA)
else:
    valid, test = splits["valid"], splits["test"]
print(f"train: {len(train)} normales · valid: {len(valid)} patches · test: {len(test)} patches")

resultados = {}   # nombre -> {split: puntajes, mapas, items}
fabricas = {}     # nombre -> función que crea el método (para reutilizar el modelo guardado en test)
metricas = {}     # nombre -> métricas en valid

def correr(variante, crear_metodo, config, normales=train):
    # Ajusta (o reutiliza) un método, lo evalúa en valid y muestra sus métricas.
    fabricas[variante] = crear_metodo
    resultados[variante] = ab.run(nombre(variante), crear_metodo, normales, {"valid": valid},
                                  recompute=RECALCULAR, config=config)
    metricas[variante] = ab.evaluate(variante, resultados[variante]["valid"])
    return ab.metrics_table([metricas[variante]])

## 2. B0 — Estadística local (método clásico)

Para cada píxel se calcula, en una ventana de 15×15, la **media** (brillo) y el **desvío** (cuánta textura hay) con filtros de caja. Con la tela normal se estima cuánto suelen valer; un píxel es anómalo si se aparta mucho en alguno de los dos. El mapa se suaviza con una gaussiana (σ = 4). El puntaje del patch es el máximo del mapa.

In [ ]:
VENTANA_B0 = 15
correr("B0_estadistica_local", lambda: ab.LocalStatistics(window=VENTANA_B0), {"metodo": "B0", "ventana": VENTANA_B0})

## 3. B1 — PaDiM

Se concatenan las salidas de `layer1`–`layer3` de una ResNet18 preentrenada (mapa de 64×64 posiciones para un patch de 256), se eligen 100 canales al azar y se ajusta una gaussiana por posición. Un patch nuevo es anómalo en una posición si su característica está lejos de esa gaussiana (distancia de Mahalanobis).

In [ ]:
correr("B1_padim_resnet18", lambda: ab.PaDiM(backbone="resnet18", dims=100, seed=SEMILLA),
       {"metodo": "PaDiM", "backbone": "resnet18", "dims": 100})

## 4. B2 — PatchCore

1. Se extraen características de `layer2` y `layer3` de la ResNet, promediadas en vecindarios de 3×3 (cada vector describe una zona local de la tela).
2. Se juntan todas las de las imágenes normales en un **banco de memoria**.
3. Un **coreset** greedy conserva sólo el `CORESET` (1 %) de los vectores más representativos: cubre la variedad de la tela normal con mucha menos memoria y tiempo.
4. En un patch nuevo, cada zona recibe como puntaje la **distancia al vecino más cercano** del banco. El mapa se amplía a 256 y se suaviza; el puntaje del patch es el máximo.

In [ ]:
def patchcore(backbone=BACKBONE, coreset=CORESET, size=256):
    return lambda: ab.PatchCore(backbone=backbone, coreset=coreset, size=size, seed=SEMILLA)

B2 = f"B2_patchcore_{BACKBONE}_c{CORESET:g}"
correr(B2, patchcore(), {"metodo": "PatchCore", "backbone": BACKBONE, "coreset": CORESET, "resolucion": 256})

### Comparación de los tres métodos en valid

In [ ]:
base = ["B0_estadistica_local", "B1_padim_resnet18", B2]
display(ab.metrics_table([metricas[n] for n in base]))
display(ab.roll_table([metricas[n] for n in base]))
ab.plot_roc(resultados, base);

In [ ]:
ab.plot_maps(resultados, base, n=5);

## 5. Ablaciones de PatchCore (sólo en valid)

Se cambia **un factor por vez** respecto de la configuración base (`BACKBONE`, `CORESET`, `N_NORMALES`, resolución 256):

1. **Backbone:** WideResNet50 (el del paper) vs. ResNet18. ¿Vale la pena una red ~6× más pesada?
2. **Coreset:** 1 % vs. 10 %. Más memoria, más tiempo de búsqueda; ¿mejora?
3. **Cantidad de normales:** ¿cuántos patches sanos hacen falta? Responde cuánto dato normal requiere el paradigma.
4. **Resolución:** patches reducidos a 128 vs. 256 nativos. Es la misma pregunta de escala del bloque A, en el paradigma de anomalías.

> En CPU, WideResNet50 y el coreset de 10 % son las corridas más lentas. Se pueden comentar si el tiempo no alcanza.

In [ ]:
ABL_BACKBONE = ["wide_resnet50_2"]
ABL_CORESET = [0.1]
ABL_NORMALES = [100, 2000] if not MODO_PRUEBA else [20]
ABL_RESOLUCION = [128]

ablaciones = [B2]
for backbone in ABL_BACKBONE:
    variante = f"B2_patchcore_{backbone}_c{CORESET:g}"
    correr(variante, patchcore(backbone=backbone), {"metodo": "PatchCore", "backbone": backbone, "coreset": CORESET})
    ablaciones.append(variante)
for coreset in ABL_CORESET:
    variante = f"B2_patchcore_{BACKBONE}_c{coreset:g}"
    correr(variante, patchcore(coreset=coreset), {"metodo": "PatchCore", "backbone": BACKBONE, "coreset": coreset})
    ablaciones.append(variante)
for n in ABL_NORMALES:
    variante = f"B2_patchcore_{BACKBONE}_c{CORESET:g}_n{n}"
    correr(variante, patchcore(), {"metodo": "PatchCore", "normales": n},
           normales=ab.sample_items(splits["train"], n, seed=SEMILLA))
    ablaciones.append(variante)
for size in ABL_RESOLUCION:
    variante = f"B2_patchcore_{BACKBONE}_c{CORESET:g}_r{size}"
    correr(variante, patchcore(size=size), {"metodo": "PatchCore", "resolucion": size})
    ablaciones.append(variante)

display(ab.metrics_table([metricas[n] for n in ablaciones]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ab.plot_cost_vs_quality([metricas[n] for n in base + ablaciones[1:]], ax=axes[0])
ab.plot_roc(resultados, ablaciones, ax=axes[1])
plt.tight_layout()

## 6. Análisis de errores (valid)

Defectos con **menor** puntaje según la mejor variante de PatchCore: los que más le cuesta detectar. Conviene mirar si son defectos chicos, de bajo contraste o de algún rollo en particular.

In [ ]:
candidatas = [n for n in metricas if n.startswith("B2_")]
mejor = max(candidatas, key=lambda n: (metricas[n]["AUROC_patch"], -(metricas[n]["segundos_por_patch"] or 0)))
print("Mejor PatchCore en valid:", mejor)
ab.plot_maps(resultados, [mejor, "B1_padim_resnet18", "B0_estadistica_local"], n=5, worst=True);

## 7. Evaluación única en test

**Criterio fijado antes de mirar test:** la variante de PatchCore con mayor **AUROC por patch en valid** (empate → la más rápida). En test se evalúan B0, B1 y esa variante, cada una con **su umbral de valid**. Si los intervalos se superponen, no se puede afirmar que un método sea mejor.

In [ ]:
if EVALUAR_TEST:
    finales = []
    for variante in ("B0_estadistica_local", "B1_padim_resnet18", mejor):
        r = ab.run(nombre(variante), fabricas[variante], None, {"test": test})   # reutiliza el modelo guardado
        finales.append(ab.evaluate(variante, r["test"], threshold=metricas[variante]["umbral"]))
        resultados[variante]["test"] = r["test"]
    display(ab.metrics_table(finales))
    display(ab.roll_table(finales))
    ab.plot_roc(resultados, [m["corrida"] for m in finales], split="test")
else:
    print("Test no evaluado (EVALUAR_TEST = False).")

## 8. Conclusiones

> Completar **después** de correr sin `MODO_PRUEBA`, con los números de las tablas.

- **B0 vs. B1/B2:** ¿cuánto aportan las características de una CNN preentrenada frente a estadísticas de brillo y textura? ¿En qué defectos el método clásico alcanza?
- **PaDiM vs. PatchCore:** ¿se confirma que modelar por posición rinde peor en una textura? ¿Se ve en los mapas?
- **Ablaciones:** ¿qué decisión pesa más (backbone, coreset, normales, resolución) y a qué costo?
- **Por rollo:** ¿hay rollos donde todos fallan? ¿Por qué?
- **Limitaciones:** patches de una misma madre no son independientes (por eso el bootstrap por madre); los `test/good` vienen de madres defectuosas; es tela industrial, no fotos de prendas.

Este es el **segundo paradigma** del proyecto. La comparación directa con un modelo supervisado se hace en FabriD (bloque C), donde ambos se pueden evaluar sobre el mismo test.